In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('clothing_reviews.csv')

In [3]:
df.shape

(23486, 11)

In [4]:
df.head()

,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23486 entries, 0 to 23485
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Unnamed: 0               23486 non-null  int64 
 1   Clothing ID              23486 non-null  int64 
 2   Age                      23486 non-null  int64 
 3   Title                    19676 non-null  object
 4   Review Text              22641 non-null  object
 5   Rating                   23486 non-null  int64 
 6   Recommended IND          23486 non-null  int64 
 7   Positive Feedback Count  23486 non-null  int64 
 8   Division Name            23472 non-null  object
 9   Department Name          23472 non-null  object
 10  Class Name               23472 non-null  object
dtypes: int64(6), object(5)
memory usage: 2.0+ MB


In [8]:
df = df.dropna(subset=['Review Text'], axis=0)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 22641 entries, 0 to 23485
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Unnamed: 0               22641 non-null  int64 
 1   Clothing ID              22641 non-null  int64 
 2   Age                      22641 non-null  int64 
 3   Title                    19675 non-null  object
 4   Review Text              22641 non-null  object
 5   Rating                   22641 non-null  int64 
 6   Recommended IND          22641 non-null  int64 
 7   Positive Feedback Count  22641 non-null  int64 
 8   Division Name            22628 non-null  object
 9   Department Name          22628 non-null  object
 10  Class Name               22628 non-null  object
dtypes: int64(6), object(5)
memory usage: 2.1+ MB


In [10]:
df['Rating'].unique()

array([4, 5, 3, 2, 1])

In [11]:
df['Rating'].value_counts()

Rating
5    12540
4     4908
3     2823
2     1549
1      821
Name: count, dtype: int64

In [12]:
df['RatingSentiment'] = df['Rating'].map({1:'negative', 2:'negative', 3:'neutral', 4:'positive', 5:'positive'})

/var/folders/sk/0kd2ktm55z3_wkvb9bfbq3zr0000gn/T/ipykernel_31306/3770533187.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['RatingSentiment'] = df['Rating'].map({1:'negative', 2:'negative', 3:'neutral', 4:'positive', 5:'positive'})


In [13]:
df['RatingSentiment'].value_counts()

RatingSentiment
positive    17448
neutral      2823
negative     2370
Name: count, dtype: int64

In [38]:
from textblob import TextBlob

In [39]:
df['polarity'] = df['Review Text'].apply(lambda review : TextBlob(review).sentiment.polarity)

In [40]:
df['polarity'].min()

-0.9750000000000001

In [41]:
df['PredictedSentiment'] = df['polarity'].apply(lambda p : 'positive' if p>0.05 else ('negative' if p<-0.05 else 'neutral'))

In [42]:
df['PredictedSentiment'].value_counts()

PredictedSentiment
positive    20276
neutral      1601
negative      764
Name: count, dtype: int64

In [43]:
textblob_accuracy = (df['PredictedSentiment'] == df['RatingSentiment']).mean()

In [44]:
textblob_accuracy

0.7593304182677444

## MACHINE LEARNING APPROACH

In [45]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

In [46]:
vectorizer = TfidfVectorizer()

In [47]:
X = vectorizer.fit_transform(df['Review Text'])

In [49]:
y = df['RatingSentiment']

In [50]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [51]:
model = RandomForestClassifier()

In [52]:
model.fit(X_train, y_train)

RandomForestClassifier()

In [53]:
prediction = model.predict(X_test)

In [55]:
accuracy = accuracy_score(y_test, prediction)

In [56]:
accuracy

0.7734599249282402